In [11]:
!pip install boto3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 2.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 1.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 6.2 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Helper para criação de tabelas
from pyiceberg.partitioning import PartitionSpec
from pyiceberg.transforms import IdentityTransform
from pyiceberg.schema import Schema
from pyiceberg.io.pyarrow import pyarrow_to_schema
from pyiceberg.catalog import load_catalog
from pyiceberg.types import NestedField, StringType
from datetime import datetime
import polars as pl
import boto3
from io import BytesIO

# -------------------
# catálogo
# -------------------
catalog = load_catalog(
    "dev",
    uri="http://rest:8181",

    **{
        "s3.endpoint": "http://minio:9000",
        "s3.access-key-id": "minioadmin",
        "s3.secret-access-key": "minioadmin",
        "s3.path-style-access": "true",
    }
)

# -------------------
# s3 / minio
# -------------------
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
)



def build_partition_spec(schema, partition_cols):
    # nenhum particionamento
    if not partition_cols:
        return PartitionSpec()

    fields = []

    for i, col in enumerate(partition_cols, start=1000):
        field = schema.find_field(col)

        if field is None:
            raise ValueError(f"Coluna '{col}' não existe no schema")

        fields.append(
            PartitionSpec.Field(
                source_id=field.field_id,
                field_id=i,
                transform=IdentityTransform(),
                name=col,
            )
        )

    return PartitionSpec(*fields)

def update_schema(table, arrow_schema):
    """
    Atualiza o schema da tabela Iceberg se houver colunas novas no DataFrame.
    
    Parameters
    ----------
    table : Iceberg table
        Tabela a ser verificada/atualizada
    arrow_schema : pyarrow.Schema
        Schema do DataFrame que será inserido
        
    Returns
    -------
    bool
        True se o schema foi atualizado, False caso contrário
    """ 
    # Atualiza o schema adicionando as novas colunas
    with table.update_schema() as update:
        update.union_by_name(arrow_schema)
    
    print(f"✔ schema atualizado")
    return True

def add_ingestion_metadata(df, file): 
    return df.with_columns(
        pl.lit(file).alias("_source_file"),
        pl.lit(datetime.utcnow().isoformat()).alias("_ingestion_time"), 
    )

def set_table(
    catalog,
    table_name: str,
    df,
    partition_cols=None,
    mode="append",
):
    """
    Cria ou carrega uma tabela Iceberg e escreve dados nela.

    Parameters
    ----------
    catalog : Iceberg catalog
    table_name : str
    df : polars.DataFrame
    partition_cols : list[str] | None
    mode : 'append' | 'overwrite'
    """

    # garante nomes limpos
    df = df.rename(lambda c: c.strip())

    arrow_table = df.to_arrow()
    arrow_schema = arrow_table.schema


    # -------------------
    # load or create
    # -------------------
    try:
        table = catalog.load_table(table_name)
        update_schema(table, arrow_schema)
        print(f"✔ tabela '{table_name}' carregada")

    except Exception as e:
        if "NoSuchTable" not in str(type(e)):
            raise  # erro real → não cria tabela
    
        print(f"⚙ criando tabela '{table_name}'")
    
        partition_spec = build_partition_spec(arrow_schema, partition_cols)
    
        table = catalog.create_table(
            identifier=table_name,
            schema=arrow_schema,
            partition_spec=partition_spec,
            properties={
                "write.format.default": "parquet",
                "write.parquet.compression-codec": "zstd",
                "schema.name-mapping.default": "true",
            },
        )
    # -------------------
    # write
    # -------------------
    if mode == "overwrite":
        table.overwrite(arrow_table)
    elif mode == "append":
        table.append(arrow_table)
    elif mode in ("reset"):
        print("⚠ full reset da tabela")
        table.delete("true")   # remove todos datafiles ativos
        table.append(arrow_table)

    print(f"✔ dados gravados ({mode})")

    return table


ModuleNotFoundError: No module named 'polars'

In [97]:
bucket = "raw"
candle_file_list = [
  "cryptocurrency/BTCUSDT-5m-2026-01.csv",
  "cryptocurrency/BTCUSDT-5m-2025-01.csv",
  "cryptocurrency/BTCUSDT-5m-2025-02.csv",
  "cryptocurrency/BTCUSDT-5m-2025-03.csv",
  "cryptocurrency/BTCUSDT-5m-2025-04.csv",
  "cryptocurrency/BTCUSDT-5m-2025-05.csv",
  "cryptocurrency/BTCUSDT-5m-2025-06.csv",
  "cryptocurrency/BTCUSDT-5m-2025-07.csv",
  "cryptocurrency/BTCUSDT-5m-2025-08.csv",
  "cryptocurrency/BTCUSDT-5m-2025-09.csv",
  "cryptocurrency/BTCUSDT-5m-2025-10.csv",
  "cryptocurrency/BTCUSDT-5m-2025-11.csv",
  "cryptocurrency/BTCUSDT-5m-2025-12.csv",
]

for key in candle_file_list:
    response = s3.get_object(Bucket=bucket, Key=key)
    buffer = BytesIO(response["Body"].read())
    
    df = pl.read_csv(buffer)
    # adiciona metadata de ingestão
    df = add_ingestion_metadata(df, key)
    
    arrow_table = df.to_arrow()
    table_name = 'bronze.crypto_candles'
    table = catalog.create_table_if_not_exists(table_name, schema=arrow_table.schema)
    table.overwrite(
        arrow_table, 
        overwrite_filter=EqualTo("_source_file", key)
    )

print("✔ RAW ingestão concluída")

/usr/local/lib/python3.10/site-packages/pyiceberg/table/__init__.py:651: UserWarning: Delete operation did not match any records
  warnings.warn("Delete operation did not match any records")


✔ RAW ingestão concluída


In [99]:
table = catalog.load_table("bronze.crypto_candles")
df = pl.from_arrow(table.scan().to_arrow())
df

open_time,open,high,low,close,volume,close_time,quote_volume,count,taker_buy_volume,taker_buy_quote_volume,ignore,_source_file,_ingestion_time
i64,f64,f64,f64,f64,f64,i64,f64,i64,f64,f64,i64,str,str
1764547200000,90320.5,90396.9,89228.0,89771.4,5804.585,1764547499999,5.2111e8,81147,2043.971,1.8349e8,0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T16:16:25.714790"""
1764547500000,89771.3,89783.6,88900.0,89101.0,5855.894,1764547799999,5.2288e8,115396,2413.271,2.1555e8,0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T16:16:25.714790"""
1764547800000,89100.9,89268.8,88807.2,88887.7,3430.374,1764548099999,3.0532e8,77883,1522.208,1.3552e8,0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T16:16:25.714790"""
1764548100000,88887.8,89056.2,88591.0,88673.4,3501.078,1764548399999,3.1083e8,74174,1468.706,1.3043e8,0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T16:16:25.714790"""
1764548400000,88673.5,88818.5,88374.0,88503.7,3226.298,1764548699999,2.8557e8,80373,1525.963,1.3509e8,0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T16:16:25.714790"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
1769902500000,79088.2,79148.5,78831.3,78871.1,612.724,1769902799999,4.8416e7,26495,276.025,2.1816e7,0,"""cryptocurrency/BTCUSDT-5m-2026…","""2026-02-20T16:16:08.378327"""
1769902800000,78871.1,78989.3,78724.0,78795.7,392.813,1769903099999,3.0971e7,18839,216.378,1.7059e7,0,"""cryptocurrency/BTCUSDT-5m-2026…","""2026-02-20T16:16:08.378327"""
1769903100000,78795.7,78831.9,78599.9,78706.6,520.056,1769903399999,4.0928e7,24050,261.101,2.0547e7,0,"""cryptocurrency/BTCUSDT-5m-2026…","""2026-02-20T16:16:08.378327"""


In [75]:
table = catalog.load_table("bronze.crypto_candles")
df = pl.from_arrow(table.scan().to_arrow())
df = df.rename({"open_time": "timestamp"})
df = df.with_columns(
    timestamp = pl.from_epoch(pl.col("timestamp"), time_unit="ms"),
    symbol=pl.lit("BTCUSDT"),
    timeframe=pl.lit("5m"),
)

# pl.from_epoch(pl.col("timestamp"), time_unit="s")

Schema([('timestamp', Datetime(time_unit='ms', time_zone=None)),
        ('open', Float64),
        ('high', Float64),
        ('low', Float64),
        ('close', Float64),
        ('volume', Float64),
        ('close_time', Int64),
        ('quote_volume', Float64),
        ('count', Int64),
        ('taker_buy_volume', Float64),
        ('taker_buy_quote_volume', Float64),
        ('ignore', Int64),
        ('_source_file', String),
        ('_ingestion_time', String),
        ('symbol', String),
        ('timeframe', String)])

In [100]:

from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.expressions import EqualTo, In
from pyiceberg.transforms import IdentityTransform, MonthTransform
from pyiceberg.types import TimestampType, StringType, FloatType, DoubleType
from pyiceberg.io.pyarrow import schema_to_pyarrow

schema = Schema(
    NestedField(field_id=1,  name='timestamp',              field_type=TimestampType(), required=True),
    NestedField(field_id=2,  name='symbol',                 field_type=StringType(),    required=True),
    NestedField(field_id=3,  name='timeframe',              field_type=StringType(),    required=True),
    NestedField(field_id=4,  name='open',                   field_type=FloatType(),     required=True),
    NestedField(field_id=5,  name='high',                   field_type=FloatType(),     required=True),
    NestedField(field_id=6,  name='low',                    field_type=FloatType(),     required=True),
    NestedField(field_id=7,  name='close',                  field_type=FloatType(),     required=True),
    NestedField(field_id=8,  name='volume',                 field_type=FloatType(),     required=True),
    NestedField(field_id=9,  name='close_time',             field_type=DoubleType(),    required=False),
    NestedField(field_id=10, name='quote_volume',           field_type=FloatType(),     required=False),
    NestedField(field_id=11, name='count',                  field_type=DoubleType(),    required=False),
    NestedField(field_id=12, name='taker_buy_volume',       field_type=FloatType(),     required=False),
    NestedField(field_id=13, name='taker_buy_quote_volume', field_type=FloatType(),     required=False),
    NestedField(field_id=14, name='ignore',                 field_type=DoubleType(),    required=False),
    NestedField(field_id=15, name='_source_file',           field_type=StringType(),    required=True),
    NestedField(field_id=16, name='_ingestion_time',        field_type=StringType(),    required=True),
)
partition_spec = PartitionSpec(
    # Partição por Ativo (BTCUSDT, ETHUSDT...)
    PartitionField(
        source_id=2,       # id do campo 'symbol'
        field_id=1000,     # id sequencial para a partição
        transform=IdentityTransform(), 
        name="symbol"
    ),
    # Partição por Timeframe (1m, 5m, 1h...)
    PartitionField(
        source_id=3,       # id do campo 'timeframe'
        field_id=1001, 
        transform=IdentityTransform(), 
        name="timeframe"
    ),
    # Partição Temporal por Mês (Extrai o mês do timestamp automaticamente)
    PartitionField(
        source_id=1,       # id do campo 'timestamp'
        field_id=1002, 
        transform=MonthTransform(), 
        name="month"
    )
)
silver_table_name = "silver.cripto_currency"
catalog.create_table_if_not_exists(silver_table_name, schema=schema, partition_spec=partition_spec)

table = catalog.load_table("bronze.crypto_candles")
df = pl.from_arrow(table.scan().to_arrow())
df = df.rename({"open_time": "timestamp"})
df = df.with_columns(
    timestamp = pl.from_epoch(pl.col("timestamp"), time_unit="ms"),
    symbol=pl.lit("BTCUSDT"),
    timeframe=pl.lit("5m"),
)
files = df["_source_file"].unique().to_list()
iceberg_fields = [f.name for f in silver_table.schema().fields]
df = df.select(iceberg_fields)
silver_table = catalog.load_table(silver_table_name)
arrow_schema = schema_to_pyarrow(silver_table.schema())
arrow_table = df.to_arrow().cast(arrow_schema)
silver_table.overwrite(arrow_table, overwrite_filter=In("_source_file", files))

print("✅ Novos valores corretos")

✅ Novos valores corretos


In [86]:
catalog.load_table("silver.btcusdt5m").schema()

Schema(NestedField(field_id=1, name='timestamp', field_type=StringType(), required=False), NestedField(field_id=2, name='open', field_type=DoubleType(), required=False), NestedField(field_id=3, name='high', field_type=DoubleType(), required=False), NestedField(field_id=4, name='low', field_type=DoubleType(), required=False), NestedField(field_id=5, name='close', field_type=DoubleType(), required=False), NestedField(field_id=6, name='volume', field_type=DoubleType(), required=False), NestedField(field_id=7, name='_source_file', field_type=StringType(), required=False), NestedField(field_id=8, name='_ingestion_time', field_type=StringType(), required=False), schema_id=0, identifier_field_ids=[])

In [37]:
import polars as pl
table = catalog.load_table("silver.cripto_currency")
df = pl.scan_iceberg(table, storage_options=table.catalog.properties)
df = df.filter(
    (pl.col("symbol") == "BTCUSDT")
    & (pl.col("timeframe") == "5m")
)
df = df.sort("timestamp")

df = df.with_columns(
    ema_20=pl.col("close").ewm_mean(span=20),
    ema_200=pl.col("close").ewm_mean(span=200),
    volatility_20=pl.col("close").rolling_std(window_size=20),
    rsi=((pl.col("close") - pl.col("close").shift(1)).fill_null(0)
             .pipe(lambda diff: (diff.clip(0, None).ewm_mean(span=14) / 
                                 diff.clip(None, 0).abs().ewm_mean(span=14)))
             .pipe(lambda rs: 100 - (100 / (1 + rs)))
            )
)

df.limit(30).collect()

timestamp,symbol,timeframe,open,high,low,close,volume,close_time,quote_volume,count,taker_buy_volume,taker_buy_quote_volume,ignore,_source_file,_ingestion_time,ema_20,ema_200,volatility_20,rsi
datetime[μs],str,str,f32,f32,f32,f32,f32,f64,f32,f64,f32,f32,f64,str,str,f32,f32,f32,f32
2025-01-01 00:00:00,"""BTCUSDT""","""5m""",93548.796875,93690.0,93514.203125,93648.398438,341.604004,1.7357e12,3.1982602e7,8176.0,148.317993,1.3885566e7,0.0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T22:43:14.863214+00:…",93648.398438,93648.398438,null,NaN
2025-01-01 00:05:00,"""BTCUSDT""","""5m""",93648.296875,93666.703125,93576.898438,93585.898438,166.341003,1.7357e12,1.5570568e7,4688.0,82.528999,7.724998e6,0.0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T22:43:14.863214+00:…",93615.585938,93616.992188,null,0.0
2025-01-01 00:10:00,"""BTCUSDT""","""5m""",93586.0,93637.203125,93460.203125,93631.203125,480.20401,1.7357e12,4.4908864e7,9464.0,141.093994,1.3196478e7,0.0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T22:43:14.863214+00:…",93621.320312,93621.773438,null,45.545464
2025-01-01 00:15:00,"""BTCUSDT""","""5m""",93631.203125,93829.398438,93588.0,93781.0,545.646973,1.7357e12,5.1149252e7,11216.0,330.415009,3.0975474e7,0.0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T22:43:14.863214+00:…",93667.414062,93662.179688,null,80.108742
2025-01-01 00:20:00,"""BTCUSDT""","""5m""",93781.0,93885.0,93680.0,93712.0,392.041992,1.7357e12,3.6765164e7,7673.0,204.949005,1.9220376e7,0.0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T22:43:14.863214+00:…",93678.195312,93672.34375,null,59.901299
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-01-01 02:05:00,"""BTCUSDT""","""5m""",93724.703125,93760.398438,93641.398438,93660.601562,362.049988,1.7357e12,3.392456e7,7858.0,143.636993,1.3458921e7,0.0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T22:43:14.863214+00:…",93824.578125,93855.75,194.360153,38.704372
2025-01-01 02:10:00,"""BTCUSDT""","""5m""",93660.5,93784.5,93642.296875,93768.0,149.701004,1.7357e12,1.4031057e7,4718.0,94.327003,8.840848e6,0.0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T22:43:14.863214+00:…",93818.804688,93852.0625,196.703674,47.997047
2025-01-01 02:15:00,"""BTCUSDT""","""5m""",93767.898438,93771.398438,93656.898438,93735.898438,205.520004,1.7357e12,1.925812e7,5365.0,97.862999,9.169474e6,0.0,"""cryptocurrency/BTCUSDT-5m-2025…","""2026-02-20T22:43:14.863214+00:…",93810.398438,93847.328125,200.214645,45.61216


dev (<class 'pyiceberg.catalog.rest.RestCatalog'>)